In [ ]:
import os
os.environ['IVY_HOME'] = '/home/spark/.ivy2'

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("wikimedia_analysis")
    .master("spark://spark-master:7077").config("spark.jars.ivy", "/home/spark/.ivy2")
    
    # Packages (Iceberg, Hadoop, AWS)
    .config("spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1,"
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262"
    )
    
    # Iceberg Extensions
    .config("spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    
    # S3A Endpoint & Credentials
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    
    # Timeouts (in milliseconds - NUMBERS ONLY)
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60000")
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000")
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000")
    .config("spark.hadoop.fs.s3a.socket.recv.buffer", "65536")
    
    # Retry & Multipart Settings
    .config("spark.hadoop.fs.s3a.retry.interval", "500")
    .config("spark.hadoop.fs.s3a.retry.throttle.interval", "500")
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400000")
    
    # Session & Delegation Token Duration
    .config("spark.hadoop.fs.s3a.assumed.role.session.duration", "3600000")
    .config("spark.hadoop.fs.s3a.delegation.token.duration", "3600000")
    
    .getOrCreate()
)

In [ ]:
df = spark.read.format("iceberg").load(
    "s3a://soc/raw/recent_changes"
)

In [ ]:
df.printSchema()

In [ ]:
df.count()